# Week 4 – Predictive Modeling and Optimization in Logistics Systems

## Project: Logistics Delivery-Time Prediction

### Objective
Build a regression model to predict shipment delivery time and use the predictions to propose practical logistics optimization strategies.

### Dataset
`logistics_delivery_prediction_dataset.csv`

> The dataset is synthetic and is used for educational demonstration. It does not represent confidential or actual company operational data.


## 1. Business Problem

Logistics delivery time is affected by transportation distance, traffic, weather, hub congestion, shipment priority, number of stops, vehicle age, warehouse processing time, and day of the week.

The goal of this project is to predict **Delivery_Time_Hours** using these shipment and operational features.

### Business questions
1. Can machine learning predict delivery time?
2. Which regression model performs best on the available data?
3. Which operational variables are most useful for prediction?
4. How can predictions be converted into logistics optimization actions?


In [1]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("Libraries imported successfully.")


Libraries imported successfully.


## 2. Load the Dataset

The CSV is stored in the `data` folder. Because this notebook is inside the `notebooks` folder, `../data/` moves one level up and then into the data folder.


In [2]:
# Load the logistics dataset
file_path = "../data/logistics_delivery_prediction_dataset.csv"

df = pd.read_csv(file_path)

print("Dataset loaded successfully.")
print("Shape:", df.shape)
df.head()


FileNotFoundError: [Errno 2] No such file or directory: '../data/logistics_delivery_prediction_dataset.csv'

## 3. Understand the Dataset

The dataset contains shipment-level observations.

### Target variable
`Delivery_Time_Hours`

### Predictor variables
- Distance_km
- Package_Weight_kg
- Traffic_Level
- Weather
- Hub_Load
- Priority
- Stops
- Vehicle_Age_Years
- Warehouse_Processing_Hours
- Day_of_Week


In [ ]:
# Basic dataset information
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)


In [ ]:
# Statistical summary for numeric columns
df.describe()


In [ ]:
# Check for missing values
missing_values = df.isnull().sum()

print("Missing values by column:")
print(missing_values)

print("\nTotal missing values:", missing_values.sum())


## 4. Exploratory Data Analysis

Exploratory Data Analysis (EDA) helps us understand the data before building the machine-learning model.

We will inspect:
- Delivery-time distribution
- Traffic and delivery time
- Distance and delivery time
- Weather conditions
- Shipment priority


In [ ]:
# Distribution of delivery time
plt.figure(figsize=(8, 5))
plt.hist(df["Delivery_Time_Hours"], bins=30)
plt.xlabel("Delivery Time (Hours)")
plt.ylabel("Number of Shipments")
plt.title("Distribution of Delivery Time")
plt.tight_layout()
plt.show()


In [ ]:
# Average delivery time by traffic level
traffic_summary = df.groupby("Traffic_Level")["Delivery_Time_Hours"].mean()

plt.figure(figsize=(8, 5))
plt.plot(traffic_summary.index, traffic_summary.values, marker="o")
plt.xlabel("Traffic Level")
plt.ylabel("Average Delivery Time (Hours)")
plt.title("Traffic Level vs Average Delivery Time")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# Distance vs delivery time
plt.figure(figsize=(8, 5))
plt.scatter(df["Distance_km"], df["Delivery_Time_Hours"], alpha=0.35)
plt.xlabel("Distance (km)")
plt.ylabel("Delivery Time (Hours)")
plt.title("Distance vs Delivery Time")
plt.tight_layout()
plt.show()


In [ ]:
# Average delivery time by weather
weather_summary = df.groupby("Weather")["Delivery_Time_Hours"].mean().sort_values()

plt.figure(figsize=(8, 5))
plt.bar(weather_summary.index, weather_summary.values)
plt.xlabel("Weather")
plt.ylabel("Average Delivery Time (Hours)")
plt.title("Weather vs Average Delivery Time")
plt.tight_layout()
plt.show()


## 5. Prepare Features and Target

Machine learning separates the data into:

- **X:** predictor variables used by the model.
- **y:** target variable that the model must predict.

The target `Delivery_Time_Hours` must not be included inside X because that would leak the answer to the model.


In [ ]:
# Separate predictors and target
X = df.drop(columns=["Delivery_Time_Hours"])
y = df["Delivery_Time_Hours"]

print("X shape:", X.shape)
print("y shape:", y.shape)


## 6. Identify Numeric and Categorical Variables

Machine-learning algorithms require numerical inputs.

Numeric columns can be passed through directly, while categorical columns such as Weather, Priority, and Day_of_Week are converted using one-hot encoding.


In [ ]:
categorical_columns = ["Weather", "Priority", "Day_of_Week"]
numeric_columns = [column for column in X.columns if column not in categorical_columns]

print("Numeric columns:")
print(numeric_columns)

print("\nCategorical columns:")
print(categorical_columns)


In [ ]:
# Create preprocessing pipeline
preprocess = ColumnTransformer([
    ("num", "passthrough", numeric_columns),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_columns)
])

print("Preprocessing pipeline created.")


## 7. Train-Test Split

The dataset is divided into:

- **80% training data** – used to learn patterns.
- **20% test data** – used to evaluate performance on unseen observations.

`random_state=42` makes the split reproducible.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))


## 8. Model 1 – Linear Regression

Linear Regression is used as a baseline.

It attempts to estimate delivery time as a combination of the input variables. It is simple, fast, and relatively easy to interpret.


In [ ]:
linear_model = Pipeline([
    ("prep", preprocess),
    ("model", LinearRegression())
])

linear_model.fit(X_train, y_train)

linear_predictions = linear_model.predict(X_test)

linear_mae = mean_absolute_error(y_test, linear_predictions)
linear_rmse = np.sqrt(mean_squared_error(y_test, linear_predictions))
linear_r2 = r2_score(y_test, linear_predictions)

print("Linear Regression")
print("MAE :", round(linear_mae, 3))
print("RMSE:", round(linear_rmse, 3))
print("R²  :", round(linear_r2, 3))


## 9. Model 2 – Decision Tree Regression

A Decision Tree learns a sequence of rules and can capture nonlinear relationships.

However, if the tree becomes too complex, it can overfit the training data.


In [ ]:
tree_model = Pipeline([
    ("prep", preprocess),
    ("model", DecisionTreeRegressor(
        max_depth=8,
        random_state=42
    ))
])

tree_model.fit(X_train, y_train)

tree_predictions = tree_model.predict(X_test)

tree_mae = mean_absolute_error(y_test, tree_predictions)
tree_rmse = np.sqrt(mean_squared_error(y_test, tree_predictions))
tree_r2 = r2_score(y_test, tree_predictions)

print("Decision Tree Regression")
print("MAE :", round(tree_mae, 3))
print("RMSE:", round(tree_rmse, 3))
print("R²  :", round(tree_r2, 3))


## 10. Model 3 – Random Forest Regression

Random Forest combines many decision trees.

It is useful when relationships between logistics variables are nonlinear or involve interactions. We will first train a baseline Random Forest and later tune its hyperparameters.


In [ ]:
random_forest_model = Pipeline([
    ("prep", preprocess),
    ("model", RandomForestRegressor(
        n_estimators=180,
        max_depth=14,
        min_samples_leaf=3,
        random_state=42,
        n_jobs=-1
    ))
])

random_forest_model.fit(X_train, y_train)

rf_predictions = random_forest_model.predict(X_test)

rf_mae = mean_absolute_error(y_test, rf_predictions)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_predictions))
rf_r2 = r2_score(y_test, rf_predictions)

print("Random Forest Regression")
print("MAE :", round(rf_mae, 3))
print("RMSE:", round(rf_rmse, 3))
print("R²  :", round(rf_r2, 3))


## 11. Compare the Models

Lower MAE and RMSE indicate smaller prediction errors.

A higher R² indicates that the model explains more variation in the target under the conditions of this dataset.


In [ ]:
results = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Decision Tree",
        "Random Forest"
    ],
    "MAE (hours)": [
        linear_mae,
        tree_mae,
        rf_mae
    ],
    "RMSE (hours)": [
        linear_rmse,
        tree_rmse,
        rf_rmse
    ],
    "R²": [
        linear_r2,
        tree_r2,
        rf_r2
    ]
})

results = results.sort_values("RMSE (hours)")
results.round(3)


In [ ]:
# Visual comparison of RMSE
plt.figure(figsize=(8, 5))
plt.bar(results["Model"], results["RMSE (hours)"])
plt.xlabel("Model")
plt.ylabel("RMSE (Hours)")
plt.title("Model Performance Comparison")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()


## 12. Hyperparameter Tuning

A model's hyperparameters control how the algorithm is configured.

For Random Forest, we will test:
- `n_estimators`: number of trees
- `max_depth`: maximum depth of trees
- `min_samples_leaf`: minimum observations in a leaf

Grid search evaluates different combinations using cross-validation.


In [ ]:
# Random Forest pipeline for tuning
rf_pipeline = Pipeline([
    ("prep", preprocess),
    ("model", RandomForestRegressor(
        random_state=42,
        n_jobs=-1
    ))
])

param_grid = {
    "model__n_estimators": [120, 180],
    "model__max_depth": [10, 14, None],
    "model__min_samples_leaf": [2, 3]
}

grid_search = GridSearchCV(
    rf_pipeline,
    param_grid,
    cv=3,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("Best parameters:")
print(grid_search.best_params_)


In [ ]:
# Evaluate tuned Random Forest
tuned_model = grid_search.best_estimator_
tuned_predictions = tuned_model.predict(X_test)

tuned_mae = mean_absolute_error(y_test, tuned_predictions)
tuned_rmse = np.sqrt(mean_squared_error(y_test, tuned_predictions))
tuned_r2 = r2_score(y_test, tuned_predictions)

print("Tuned Random Forest")
print("MAE :", round(tuned_mae, 3))
print("RMSE:", round(tuned_rmse, 3))
print("R²  :", round(tuned_r2, 3))


## 13. Five-Fold Cross-Validation

Cross-validation checks whether the model performs consistently across different portions of the dataset.

In 5-fold cross-validation:
1. The data is divided into five parts.
2. Four parts are used for training.
3. One part is used for validation.
4. The process is repeated five times.
5. The results are averaged.


In [ ]:
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_rmse = np.sqrt(-cross_val_score(
    tuned_model,
    X,
    y,
    cv=cv,
    scoring="neg_mean_squared_error",
    n_jobs=-1
))

cv_mae = -cross_val_score(
    tuned_model,
    X,
    y,
    cv=cv,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)

print("5-Fold Cross-Validation")
print("Mean RMSE:", round(cv_rmse.mean(), 3))
print("RMSE SD  :", round(cv_rmse.std(), 3))
print("Mean MAE :", round(cv_mae.mean(), 3))
print("MAE SD   :", round(cv_mae.std(), 3))


## 14. Feature Importance

Feature importance shows which transformed variables contributed most to the Random Forest predictions.

Important features should be treated as predictive signals, not as proof that one variable directly causes delivery delays.


In [ ]:
feature_names = tuned_model.named_steps["prep"].get_feature_names_out()
importances = tuned_model.named_steps["model"].feature_importances_

feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
}).sort_values("Importance", ascending=False)

feature_importance.head(10)


In [ ]:
# Plot top 10 feature importances
top_features = feature_importance.head(10).sort_values("Importance")

plt.figure(figsize=(9, 6))
plt.barh(top_features["Feature"], top_features["Importance"])
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Top Predictive Features")
plt.tight_layout()
plt.show()


## 15. Predict Delivery Time for New Shipments

The trained model can be used to estimate delivery time for new shipment records.

The example below creates a few hypothetical shipments.


In [ ]:
new_shipments = pd.DataFrame({
    "Distance_km": [80, 220, 150],
    "Package_Weight_kg": [2.5, 8.0, 4.0],
    "Traffic_Level": [2, 5, 3],
    "Weather": ["Clear", "Rain", "Cloudy"],
    "Hub_Load": [0.50, 0.90, 0.70],
    "Priority": ["Express", "Standard", "Standard"],
    "Stops": [3, 8, 5],
    "Vehicle_Age_Years": [2, 7, 4],
    "Warehouse_Processing_Hours": [1.5, 4.0, 2.5],
    "Day_of_Week": ["Tue", "Fri", "Wed"]
})

new_predictions = tuned_model.predict(new_shipments)

prediction_output = new_shipments.copy()
prediction_output["Predicted_Delivery_Time_Hours"] = new_predictions.round(2)

prediction_output


## 16. Delay-Risk Identification

Prediction can be converted into a simple operational risk flag.

The threshold below is **illustrative only**. In a real company, the threshold should be based on the relevant route-level SLA and historical service performance.


In [ ]:
# Illustrative threshold
SLA_THRESHOLD = 30

prediction_output["Delay_Risk"] = np.where(
    prediction_output["Predicted_Delivery_Time_Hours"] > SLA_THRESHOLD,
    "High Risk",
    "Normal"
)

prediction_output


## 17. Optimization Strategies

The model can support the following operational decisions:

### Dynamic Capacity Allocation
High-risk shipments can receive additional capacity when the expected service benefit justifies the cost.

### Hub Load Balancing
High predicted delivery times combined with high hub utilization can trigger a review of workload distribution.

### Route Prioritization
Shipments with high predicted delivery time or strict service commitments can receive additional routing attention.

### Proactive Exception Management
Potentially delayed shipments can be identified before they become confirmed service failures.

### Resource Scheduling
Predicted workload can help plan drivers, vehicles, and warehouse staffing.

### Cost Optimization
Additional resources should be allocated by balancing operating cost against delay risk and service impact.


## 18. Optimization Decision Logic

The machine-learning model predicts what may happen. An optimization or decision layer then determines what action should be considered.

A production system could follow:

**Shipment Data → Prediction → Risk Score → Constraints → Optimization Decision → Operational Action → Outcome Feedback**


In [ ]:
# Example: identify high-risk test shipments
test_results = X_test.copy()
test_results["Actual_Delivery_Time_Hours"] = y_test.values
test_results["Predicted_Delivery_Time_Hours"] = tuned_predictions

test_results["Delay_Risk"] = np.where(
    test_results["Predicted_Delivery_Time_Hours"] > SLA_THRESHOLD,
    "High Risk",
    "Normal"
)

high_risk_shipments = test_results[
    test_results["Delay_Risk"] == "High Risk"
]

print("High-risk shipments:", len(high_risk_shipments))
print("Total test shipments:", len(test_results))
print("High-risk percentage:",
      round(len(high_risk_shipments) / len(test_results) * 100, 2), "%")

high_risk_shipments.head()


## 19. Business Interpretation

The project shows how predictive analytics can change logistics decision-making from reactive to proactive.

Instead of waiting for a shipment to become late, the organization can:
1. Predict delivery time.
2. Identify high-risk shipments.
3. Investigate the main operational conditions.
4. Consider alternative actions.
5. Measure whether the action improved performance.

The model is therefore a **decision-support tool**, not a replacement for operational judgment.


## 20. Limitations

- The dataset is synthetic.
- Real GPS and road-network information is not included.
- Real-time traffic feeds are not included.
- Real SLA data is not available.
- Optimization savings are not measured using a real operational pilot.
- Feature importance does not establish causation.
- Real deployment requires monitoring for data drift and model drift.


## 21. Future Improvements

A production version could add:

- Real historical shipment data
- GPS and route information
- Real-time traffic
- Weather APIs
- Hub scan events
- Vehicle capacity
- Driver availability
- Fuel and transportation cost
- Route optimization
- Power BI/Tableau dashboards
- Real-time prediction APIs
- Automated model retraining


## 22. Conclusion

This project demonstrates a complete predictive analytics workflow for logistics delivery-time forecasting.

A simulated dataset was prepared, multiple regression models were compared, Random Forest was tuned using cross-validation, and performance was evaluated using MAE, RMSE, and R².

The resulting predictions can support dynamic capacity allocation, hub balancing, route prioritization, proactive exception management, and cost-aware resource scheduling.

The main lesson is that predictive modeling creates business value when predictions are connected to practical operational decisions and validated using real-world outcomes.


## 23. Viva / Interview Quick Answers

**Q: Why is this a regression problem?**  
A: Delivery time is a continuous numeric value, so we need to predict a number.

**Q: Why did you use MAE?**  
A: MAE gives the average prediction error in hours and is easy to interpret.

**Q: Why use RMSE?**  
A: RMSE penalizes large errors more strongly, which is useful when large delivery-time errors matter.

**Q: Why use Random Forest?**  
A: It can capture nonlinear relationships and interactions among logistics variables.

**Q: What is cross-validation?**  
A: It evaluates a model across multiple train-validation splits to obtain a more stable performance estimate.

**Q: What is hyperparameter tuning?**  
A: It tests different model settings and selects a configuration based on validation performance.

**Q: What is the difference between prediction and optimization?**  
A: Prediction estimates what will happen; optimization selects actions under objectives and constraints.

**Q: Can this model be directly deployed in a real company?**  
A: No. The current project uses synthetic data. Real deployment requires company data, validation, operational constraints, monitoring, and a controlled pilot.
